In [ ]:
import sys
from pathlib import Path

# Ensure repo root is on sys.path so `from tools...` imports work from any notebook subfolder.
_p = Path.cwd().resolve()
for _parent in [_p, *_p.parents]:
    if (_parent / 'tools' / 'search_tools.py').exists():
        sys.path.insert(0, str(_parent))
        break
del _p, _parent


In [1]:
%%capture --no-stderr
# %pip install "autogen-agentchat~=0.2.3"

# In Your OAI_CONFIG_LIST file, you must have two configs,
# one with:           "response_format": { "type": "text" }
# and the other with: "response_format": { "type": "json_object" }


[
    {"model": "gpt-4o-mini", "sk-REDACTED": "key go here", "response_format": {"type": "text"}},
]

In [2]:
import autogen
import os
from autogen.agentchat import UserProxyAgent
from autogen.agentchat.assistant_agent import AssistantAgent
from autogen.agentchat.groupchat import GroupChat
os.environ["SERPER_API_KEY"] = "1edefaec0732d11db50b993ba60539510cc55334"
from tools.search_tools import SearchTools




In [3]:
from autogen import ConversableAgent
from autogen import register_function

import os
import json
import requests

def search_internet(query: str) -> str:
        """Useful to search the internet
        about a a given topic and return relevant results"""
        print("Searching the internet...")
        top_result_to_return = 5
        url = "https://google.serper.dev/search"
        payload = json.dumps(
            {"q": query, "num": top_result_to_return, "tbm": "nws"})
        headers = {
            'X-API-KEY': os.environ['SERPER_API_KEY'],
            'content-type': 'application/json'
        }
        response = requests.request("POST", url, headers=headers, data=payload)
        # check if there is an organic key
        if 'organic' not in response.json():
            return "Sorry, I couldn't find anything about that, there could be an error with you serper api key."
        else:
            results = response.json()['organic']
            string = []
            print("Results:", results[:top_result_to_return])
            for result in results[:top_result_to_return]:
                try:
                    # Attempt to extract the date
                    date = result.get('date', 'Date not available')
                    string.append('\n'.join([
                        f"Title: {result['title']}",
                        f"Link: {result['link']}",
                        f"Date: {date}",  # Include the date in the output
                        f"Snippet: {result['snippet']}",
                        "\n-----------------"
                    ]))
                except KeyError:
                    next

            return '\n'.join(string)

        



In [4]:
import asyncio
import autogen
import os
import httpx
from typing import Optional, List, Dict, Tuple, Union
import random  # noqa E402

import matplotlib.pyplot as plt  # noqa E402
import networkx as nx  # noqa E402

import autogen  # noqa E402
from autogen.agentchat.conversable_agent import ConversableAgent  # noqa E402
from autogen.agentchat.assistant_agent import AssistantAgent  # noqa E402
from autogen.agentchat.groupchat import GroupChat  # noqa E402
from autogen.graph_utils import visualize_speaker_transitions_dict 

# Set your OpenAI API key
os.environ["OPENAI_API_KEY"] = "sk-REDACTED"

# Define a custom HTTP client
class MyHttpClient(httpx.Client):
    def __deepcopy__(self, memo):
        return self
    
llm_config = {
    "config_list": [
        {
            "model": "qwen2.5:72b",
            "api_type": "ollama",
            "client_host": "https://7r9o21n06von58-11434.proxy.runpod.net/",
        }
    ]
}




def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False


user_proxy = autogen.UserProxyAgent(
    name="User_proxy",
    system_message="A human admin who terminates the chat when the Leader agent sends a message with 'TERMINATE' mentioned it it",
    code_execution_config=False,
    human_input_mode="NEVER",
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    llm_config=llm_config,
)

AgentA = ConversableAgent(
    name="Algebra Specialist",
    system_message=(
        "Role: Algebra Specialist in a collaborative mathematical problem-solving task.\n"
        "\n"
        "Context:\n"
        "You are an expert in algebra, responsible for analyzing and solving algebraic equations and expressions. Your task is to address the algebraic condition of the problem and share your findings with the team.\n"
        "you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all of them.\n"
        "\n"
        "Instructions:\n"
        "- solve only and only problem one\n"
        "- Share your reasoning and results in the group chat.\n"
        "- Be open to adjusting your conclusions based on input from others.\n"

        "\n"
        "Note:\n"
        "- Focus on your specialization but consider information from other agents.\n"
        "- Communicate clearly and professionally.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)


AgentB = ConversableAgent(
    name="Number theory specialist",
    system_message=(
        "Role: Number Theory Specialist in a collaborative mathematical problem-solving task.\n"
        "\n"
        "Context:\n"
        "You are an expert in number theory, focusing on prime numbers, congruences, and related concepts. Your task is to address the number theory condition of the problem and share your findings with the team.\n"
        "- you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all of them.\n"
        "\n"
        "Instructions:\n"
        "- solve only and only problem two. \n"
        "- Share your reasoning and potential values of \( N \) in the group chat.\n"
        "- Be open to adjusting your conclusions based on input from others.\n"
        "\n"
        "Note:\n"
        "- Focus on your specialization but consider information from other agents.\n"
        "- Communicate clearly and professionally.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)



AgentC = ConversableAgent(
    name="Combinatorics specialist",
    system_message=(
        "Role: Combinatorics Specialist in a collaborative mathematical problem-solving task.\n"
        "\n"
        "Context:\n"
        "You are an expert in combinatorics, dealing with combinations, permutations, and related concepts. Your task is to address the combinatorics condition of the problem and share your findings with the team.\n"
        "- you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all of them.\n"
        "\n"
        "Instructions:\n"
        "- Solve only and only problem three"
        "- Find integer values of \( N \) that satisfy the problem\n"
        "- Share your reasoning and results in the group chat.\n"
        "- Be open to adjusting your conclusions based on input from others.\n"
        "\n"
        "Note:\n"
        "- Focus on your specialization but consider information from other agents.\n"
        "- Communicate clearly and professionally.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)



AgentD = ConversableAgent(
    name="Calculus Specialist",
    system_message=(
        "Role: Calculus Specialist in a collaborative mathematical problem-solving task.\n"
        "\n"
        "Context:\n"
        "You are an expert in calculus, particularly in integration and analysis of functions. Your task is to address the calculus condition of the problem and share your findings with the team.\n"
        "- you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all of them.\n"
        "\n"
        "Instructions:\n"
        "- solve only and only problem four. \n"
        "- Share your reasoning and potential values of \( N \) in the group chat.\n"
        "- Be open to adjusting your conclusions based on input from others.\n"
        "\n"
        "Note:\n"
        "- Focus on your specialization but consider information from other agents.\n"
        "- Communicate clearly and professionally.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)

AgentE = ConversableAgent(
    name="Geometry Specialist",
    system_message=(
        "Role: Geometry Specialist in a collaborative mathematical problem-solving task.\n"
        "\n"
        "Context:\n"
        "You are an expert in geometry, focusing on properties of shapes and angles. Your task is to address the geometry condition of the problem and share your findings with the team.\n"
        "- you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all of them.\n"
        "\n"
        "Instructions:\n"
        "- solve only and only problem five. \n"
        "- Share your reasoning and potential values of \( N \) in the group chat.\n"
        "- Be open to adjusting your conclusions based on input from others.\n"
        "\n"
        "Note:\n"
        "- Focus on your specialization but consider information from other agents.\n"
        "- Communicate clearly and professionally.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)


AgentF = ConversableAgent(
    name="Leader",
    system_message=(
        "Role: Team Leader overseeing the collaborative mathematical problem-solving task.\n"
        "\n"
        "you are in a groupchat with 5 mathematical experts who are solving individual math problems and trying to find the value of N satisfying all 5 of the equations"
        "you are responsible for giving out the final answer. YOU DO NOT KNOW ANY MATHEMATICS AND CANNOT SOLVE ANY OF THE PROBLEMS SO WAIT FOR RESPONSES OF ALL EXPERTS"
        "\n"
        "Instructions:\n"
        "- ONLY GIVE THE FINAL OUTPUT ON VALUE OF N WHEN ALL AGENTS HAVE GIVEN THEIR RESPONSE. THERE ARE 5 PROBLEMS SO YOU WILL GET 5 RESPONSES ABOUT THE POSSIBLE VALUES OF N"
        "- If unable to find a integer value terminate chat with saying no integer value found"
        "- ONLY WHEN THE FINAL ANSWER is achieved include the keyword 'TERMINATE' in the final message to signal the end of the groupchat"
        "- Avoid introducing new mathematical conditions not provided in the problem.\n"
        "\n"
        "Note:\n"
        "- Ensure the final answer is accurate and justified.\n"
        "- Avoid any unrelated discussions."
    ),
    llm_config=llm_config,
)

# (Optional) Tool Executor Agent if needed
Agent5 = ConversableAgent(
    name="Tool_executor",
    system_message=(
        "Role: Tool Executor responsible for running approved tools.\n"
        "\n"
        "Guidelines:\n"
        "- Execute tasks as requested by other agents, within the scope of approved tools.\n"
        "- Provide results promptly and accurately.\n"
        "\n"
        "Note:\n"
        "- Ensure all tool usage complies with company policies.\n"
        "- Maintain clear communication with requesting agents."
    ),
    llm_config=llm_config,
)




In [5]:
# import requests
# import json

# def query_ollama(prompt, model="qwen2.5:72b"):
#     url = "https://3s8bwscehzgw0q-11434.proxy.runpod.net/api/generate"  # Ensure correct endpoint
#     payload = {"model": model, "prompt": prompt}
    
#     try:
#         response = requests.post(url, json=payload)
#         response.raise_for_status()  # Check for HTTP errors
        
#         # Process response line by line
#         result = ""
#         for line in response.text.splitlines():
#             try:
#                 line_data = json.loads(line)
#                 result += line_data.get("response", "")
#                 if line_data.get("done", False):
#                     break
#             except json.JSONDecodeError:
#                 continue  # Ignore lines that aren't valid JSON
                
#         return result.strip()  # Return the concatenated response
#     except requests.exceptions.RequestException as e:
#         return {"error": "Request failed", "details": str(e)}

In [6]:
# # Add a global or class-level variable to track the first call
# is_first_call = True  # This flag tracks if the function is being called for the first time

# def custom_speaker_selection_func(last_speaker, groupchat):
#     global is_first_call

#     # If this is the first call, return the leader agent
#     if is_first_call:
#         is_first_call = False  # Reset the flag after the first call
#         print("First call detected. Setting speaker to Leader agent.")
#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Agent3" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Access the last message in the group chat
#     last_message = groupchat.messages[-1]
#     print(f"Last message content: {last_message}")

#     # Prepare the input for the LLM
#     prompt = (
#     "You are a conversation coordinator. Based on the last message, decide which agent should speak next out of the following Summarizer_Agent_1, Summarizer_Agent_2, Summarizer_Agent_3 and Leader. "
#     "ONLY RESPOND WITH THE NAME OF THE AGENT AND NOTHING ELSE. NO OTHER CHARACTERS SHOULD BE THERE IN YOUR MESSAGE.\n\n"
#     f"The last message is: {last_message.get('content', '')}"
#     )

#     # Analyze the message using the local LLM
#     response = query_ollama(prompt)
#     print(f"LLM response: {response}")

#     # Extract the relevant text from the response dictionary
#     next_speaker_name = response  # Replace 'text' with the correct key

#     # Find the corresponding agent in the group chat
#     for agent in groupchat.agents:
#         if agent.name == next_speaker_name:
#             return agent

#     # If no valid agent is found, return None or a default fallback
#     print(f"No valid agent found for the name: {next_speaker_name}")
#     return None


In [7]:
# # Add global variables to track the first call and the call count
# is_first_call = True  # Tracks if this is the first call
# call_count = 0        # Tracks the number of times the function has been called

# def custom_speaker_selection_func(last_speaker, groupchat):
#     global is_first_call, call_count

#     # If this is the first call, return the leader agent
#     if is_first_call:
#         is_first_call = False  # Reset the flag after the first call
#         print("First call detected. Setting speaker to Leader agent.")

#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Leader" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Increment the call count
#     call_count += 1

#     # If this is the 7th call, return the leader agent
#     if call_count % 7 == 0:
#         print(f"7th call detected (call count: {call_count}). Setting speaker to Leader agent.")
#         for agent in groupchat.agents:
#             if agent.name == "Leader":  # Replace "Leader" with the actual leader agent's name
#                 return agent
#         print("Error: Leader agent not found in the agents list.")
#         return None  # Handle the case where the leader agent is not found

#     # Access the last message in the group chat
#     last_message = groupchat.messages[-1]
#     print(f"Last message content: {last_message}")

#     # Prepare the input for the LLM
#     prompt = (
#         "You are a conversation coordinator. Based on the last message, decide which agent should speak next out of the following Production_Manager, Logistics_Manager, A/P—A/R_Manager, Customer_Relations/Marketing_Manager and Leader. "
#         "ONLY RESPOND WITH THE NAME OF THE AGENT AND NOTHING ELSE. NO OTHER CHARACTERS SHOULD BE THERE IN YOUR MESSAGE.\n\n"
#         f"The last message is: {last_message.get('content', '')}"
#     )

#     # Analyze the message using the local LLM
#     response = query_ollama(prompt)
#     print(f"LLM response: {response}")

#     # Extract the relevant text from the response dictionary
#     next_speaker_name = response.strip()  # Use .strip() to remove any extra spaces or newlines

#     # Find the corresponding agent in the group chat
#     for agent in groupchat.agents:
#         if agent.name == next_speaker_name:
#             return agent

#     # If no valid agent is found, return None or a default fallback
#     print(f"No valid agent found for the name: {next_speaker_name}")
#     return None


In [8]:
from autogen.graph_utils import visualize_speaker_transitions_dict
# Define your agents
agents = [AgentA, AgentB, AgentC, AgentD, AgentE, user_proxy, Agent5, AgentF]

# # Initialize the allowed speaker transitions dictionary
# allowed_speaker_transitions_dict = {}

# # Set up transitions for each agent
# for agent in agents:
#     if agent == Agent5:
#         # Agent5 cannot send messages to any agent
#         allowed_speaker_transitions_dict[agent] = []
#     else:
#         # Other agents can send messages to all agents except themselves and Agent5
#         allowed_speaker_transitions_dict[agent] = [
#             other_agent for other_agent in agents
#             if other_agent != agent and other_agent != Agent5
#         ]

# # Allow user_proxy to send messages to Agent5
# allowed_speaker_transitions_dict[user_proxy].append(Agent5)

# # Visualize the transitions
# visualize_speaker_transitions_dict(allowed_speaker_transitions_dict, agents)


In [9]:
def is_termination_msg(content) -> bool:
    have_content = content.get("content", None) is not None
    if have_content and "TERMINATE" in content["content"]:
        return True
    return False


group_chat = GroupChat(
    agents=agents,
    messages=[],
    max_round=50,
    # allowed_or_disallowed_speaker_transitions=allowed_speaker_transitions_dict,
    # speaker_transitions_type="allowed",
)
# Create the manager
manager = autogen.GroupChatManager(
    groupchat=group_chat,
    llm_config=llm_config,
    is_termination_msg=lambda x: x.get("content", "").find("TERMINATE") >= 0,
    code_execution_config=False,
)


In [10]:
# from autogen import register_function


# register_function(
#     search_internet,
#     caller=Leader,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent1,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent2,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent3,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

# register_function(
#     search_internet,
#     caller=Agent4,  # The assistant agent can suggest calls to the calculator.
#     executor=Agent5,  # The user proxy agent can execute the calculator calls.
#     name="search",  # By default, the function name is used as the tool name.
#     description="browse the internet for information using this tool",  # A description of the tool
# )

In [11]:
# chat_result = user_proxy.initiate_chat(Agent0, message="search internet about google. Use production Manager first")

In [13]:
# Initial message
initial_message = (
                    "Please work on findind the value of N which satisfies all 5 problems following collaboration between all experts \n"
                    "- problem one - Find possible integer values of \( N \) based on the equation \( 2x^3 - 5x^2 + x - 7 = 0 \) with \( x = \sqrt{N} \).\n"
                    "- problem two - Identify prime numbers \( N \) such that \( N \) and \( N + 2 \) are both prime (twin primes) and Ensure that \( N \equiv 3 \mod 4 \). \n"
                    "- problem three - Solve for \( N \) in the equation \( \binom{N}{3} = 364 \). \n"
                    "- problem four - Evaluate the definite integral \( \int_{0}^{\pi} \sin(Nx) \, dx = 0 \). \n"
                    "- problem five is - Determine the integer values of \( N \) such that a regular polygon with \( N \) sides has an internal angle that is an integer number of degrees. Use the formula \( \alpha = \left(1 - \frac{2}{N}\right) \times 180^\circ \) for the internal angle..\n"
)


# Initiate the conversation 
user_proxy.initiate_chat(manager, message=initial_message)

User_proxy (to chat_manager):

Please work on findind the value of N which satisfies all 5 problems following collaboration between all experts 
- problem one - Find possible integer values of \( N \) based on the equation \( 2x^3 - 5x^2 + x - 7 = 0 \) with \( x = \sqrt{N} \).
- problem two - Identify prime numbers \( N \) such that \( N \) and \( N + 2 \) are both prime (twin primes) and Ensure that \( N \equiv 3 \mod 4 \). 
- problem three - Solve for \( N \) in the equation \(inom{N}{3} = 364 \). 
- problem four - Evaluate the definite integral \( \int_{0}^{\pi} \sin(Nx) \, dx = 0 \). 
ight) 	imes 180^\circ \) for the internal angle..of \( N \) such that a regular polygon with \( N \) sides has an internal angle that is an integer number of degrees. Use the formula \( lpha = \left(1 - rac{2}{N}


--------------------------------------------------------------------------------

Next speaker: Algebra Specialist


>>>>>>>> USING AUTO REPLY...
Algebra Specialist (to chat_manager):

**

RuntimeError: Ollama exception occurred: Server disconnected without sending a response.

In [ ]:
last_message = group_chat.messages[-1] if group_chat.messages else None
if last_message:
    print("Final Message Content:", last_message['content'])

Final Message Content: **Group Chat - Number Theory Specialist's Update**

**Problem Two: Twin Primes with N ≡ 3 mod 4**

Hello Team,

As the Number Theory Specialist, I'll share my findings on Problem Two. We're seeking prime numbers \( N \) such that:

1. **Twin Primes:** Both \( N \) and \( N + 2 \) are prime.
2. **Congruence Condition:** \( N \equiv 3 \mod 4 \)

**Reasoning:**

* For \( N \equiv 3 \mod 4 \), possible values of \( N \) include 3, 7, 11, 15, ... However, not all will satisfy the twin prime condition.
* We'll examine each \( N \) to see if both \( N \) and \( N + 2 \) are prime.

**Potential Values of \( N \):**

| \( N \) | \( N + 2 \) | Both Prime? |
| --- | --- | --- |
| 3   | 5     | **Yes**     |
| 7   | 9     | No (9 is not prime) |
| 11  | 13    | **Yes**     |
| 15  | 17    | **Yes**     |

**Preliminary Conclusions for Problem Two:**

\( N \) could be: **3**, **11**, or **15** based on the twin primes condition with \( N \equiv 3 \mod 4 \).

**Next Steps & Qu

In [ ]:
def save_conversation_to_file(groupchat, filename="chat.txt"):
    """
    Save the entire conversation history to a specified file.

    Args:
        groupchat (GroupChat): The GroupChat instance containing the messages.
        filename (str): The name of the file to save the conversation history.
    """
    if not groupchat.messages:
        print("No messages in the group chat to save.")
        return

    # Compile the conversation history
    conversation_history = "\n".join(
        f"{msg['role']}: {msg['content']}" for msg in groupchat.messages
    )

    # Write the conversation history to the file
    with open(filename, "w", encoding="utf-8") as file:
        file.write(conversation_history)

    print(f"Conversation history saved to {filename}")
    
save_conversation_to_file(group_chat, filename="chat.txt")


Conversation history saved to chat.txt


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

# Initialize the Llama 3.1 model
llm = ChatOpenAI(
    model="llama3.1",
    base_url="http://44.221.48.158:11434/v1"
)

def structure_logs_with_local_llm(file_path, initial_message):
    """
    Reads chat logs from a file and generates a structured summary.

    Args:
        file_path (str): Path to the chat log file.
        initial_message (str): The initial task or prompt for context.

    Returns:
        str: The structured summary generated by the LLM.
    """
    # Read the chat logs from the file
    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            log_context = file.read()
    except FileNotFoundError:
        return "Error: The specified file was not found."
    except Exception as e:
        return f"An error occurred while reading the file: {e}"

    # Prepare the messages for the LLM
    messages = [
        {
            "role": "system",
            "content": (
                "You are a professional chat summarizer who goes through the entire chat and creates a proper summary based on the '{initial_message}'."
            )
        },
        {
            "role": "user",
            "content": (
                f"Convert the following agent logs into a structured format and into a proper summarized final output "
                f"based on the task '{initial_message}'.\n\nLogs:\n{log_context}"
            )
        }
    ]

    # Generate the structured summary using the LLM
    try:
        response = llm.invoke(messages)
        if isinstance(response, AIMessage):
            structured_summary = response.content
        else:
            structured_summary = "Unexpected response type from the model."
    except Exception as e:
        structured_summary = f"An error occurred during LLM processing: {e}"

    return structured_summary


In [ ]:
# Define the path to your chat log file and the initial task message
file_path = 'chat.txt'
initial_message = 'Design a comprehensive digital marketing course.'

# Generate the structured summary
summary = structure_logs_with_local_llm(file_path, initial_message)

# Output the summary
print(summary)

An error occurred during LLM processing: Request timed out.
